## PLUTO – Main Simulation

In this notebook, we visualize the progress in our main simulation runs.

### 0. Definitions

#### 0.1. Preamble

In [1]:
%%capture

%config InlineBackend.figure_formats = ['retina']

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib as mpl

import matplotlib.pyplot as plt

import pyPLUTO as pp

import io
import base64

from PIL import Image
from IPython.display import Image as HTML, display
import IPython.display as ipd

from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import Normalize, LogNorm
from matplotlib.ticker import ScalarFormatter, MultipleLocator
from matplotlib.patches import Wedge

from skimage.filters import threshold_otsu
from scipy.ndimage import uniform_filter

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=["steelblue", "olivedrab", "goldenrod", "firebrick", "rebeccapurple"]) 
plt.rcParams['figure.figsize'] = [8,5]
plt.rcParams['figure.constrained_layout.use'] = True
plt.rcParams['legend.frameon'] = False
plt.rcParams["xtick.minor.visible"] = True
plt.rcParams["ytick.minor.visible"] = True

plt.plot()
plt.close()

#### 0.2. Helpers

In [39]:
def compute_flux_function(D):
    Br = D.Bx1
    theta = D.x2
    r = D.x1
    integrand = Br * (r[:, None]**2) * np.sin(theta)[None, :]
    Psi = np.zeros_like(integrand)
    dtheta = np.diff(theta)
    Psi[:, 1:] = np.cumsum(
        0.5 * (integrand[:, 1:] + integrand[:, :-1]) * dtheta[None, :],
        axis=1
    )
    return Psi



def animate_density(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20
    
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
    
        im = ax.pcolormesh(
            X, Z, rho_frames[i], 
            cmap='magma', 
            norm=LogNorm(vmin=vmin, vmax=vmax), 
            shading='auto'
        )
    
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
    
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
    
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
    
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/density.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_velocity(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    rho_frames = {}
    vx_frames = {}
    vz_frames = {}
    vr_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
    
        vr = Di.vx1
        vth = Di.vx2
        vx = vr * np.sin(Theta) + vth * np.cos(Theta)
        vz = vr * np.cos(Theta) - vth * np.sin(Theta)
    
        vx_frames[i] = np.where(mask, vx, np.nan)
        vz_frames[i] = np.where(mask, vz, np.nan)
        vr_frames[i] = np.where(mask, vr, np.nan)
    
    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])
    
    stride1, stride2 = 1, 1
    
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )
        Xq = X[::stride1, ::stride2]
        Zq = Z[::stride1, ::stride2]
        Uq = vx_frames[i][::stride1, ::stride2]
        Wq = vz_frames[i][::stride1, ::stride2]
        Vrq = vr_frames[i][::stride1, ::stride2]
    
        colors = np.where(Vrq >= 0, 'r', 'b').ravel()
    
        ax.quiver(
            Xq, Zq, Uq, Wq,
            color=colors,
            scale_units='xy',
            angles='xy',
        )
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/velocity.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_field(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    rho_frames = {}
    Psi_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        rho_frames[i] = np.where(mask, Di.rho, np.nan)
        Psi_frames[i] = compute_flux_function(Di)

    vmin = np.nanmin([np.nanmin(r) for r in rho_frames.values()])
    vmax = np.nanmax([np.nanmax(r) for r in rho_frames.values()])

    global_absmax = max(np.nanmax(np.abs(p)) for p in Psi_frames.values())
    levels_pos = np.linspace(0, global_absmax, 100)
    levels_neg = -levels_pos[::-1]

    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')

        im = ax.pcolormesh(
            X, Z, rho_frames[i],
            cmap='gray',
            norm=LogNorm(vmin=vmin, vmax=vmax),
            shading='auto'
        )

        ax.contour(X, Z, Psi_frames[i], levels=levels_pos, colors='m', linestyle='-', linewidths=0.4)
        ax.contour(X, Z, Psi_frames[i], levels=levels_neg, colors='m', linestyle='-', linewidths=0.4)

        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')

        ax.legend(loc='upper right', labelcolor='w')

        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho$')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/field.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )

    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))



def animate_tracer(path, mode='truth', compare=False, panel=False):

    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    tracer_frames = {}
    disagreement_frames = {}

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        if panel:
            abs_diff = np.abs(Di.diskfrac - Di.tr1)
            n_theta = abs_diff.shape[1]
            disagreement_frames[i] = np.sum(abs_diff, axis=1) / n_theta
        else:
            if compare:
                tracer = Di.diskfrac - Di.tr1
            else:
                tracer = Di.diskfrac if mode == 'recover' else Di.tr1
            tracer_frames[i] = np.where(mask, tracer, np.nan)

    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    def _save_gif(frames, gif_name, width=750):
        gif_path = f'{path}/Storage/{gif_name}'
        frames[0].save(
            gif_path,
            format='GIF',
            save_all=True,
            append_images=frames[1:],
            duration=150,
            loop=0
        )
        ipd.display(ipd.HTML(f'<img src="{gif_path}" width="{width}">'))

    if panel:
        x1 = D_last.x1
        n1 = D_last.rho.shape[0]
        comp_frames = []
        for i in outlist:
            disagreement_pct = disagreement_frames[i] * 100.0
            total_disagreement_pct = (np.sum(disagreement_frames[i]) / n1) * 100.0
            total_agreement_pct = 100.0 - total_disagreement_pct

            fig, ax = plt.subplots(figsize=[7, 4])
            ax.fill_between(x1, disagreement_pct, -5, color='m', alpha=1/4, linewidth=0)
            ax.set_xscale('log')
            ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
            ax.xaxis.set_major_formatter(ScalarFormatter())
            ax.minorticks_off()
            ax.xaxis.set_minor_formatter(plt.NullFormatter())
            ax.set_yticks([0, 5, 10, 15, 20])
            ax.set_xlabel('R')
            ax.set_ylabel('|A – B| %')
            ax.set_ylim(-5, 20)

            if 'STAR' in path:
                ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
            elif 'BH' in path:
                ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
            ax.legend(loc='upper right')

            ax.text(0.5, 0.05, f'1 – |A – B| = {total_agreement_pct:.1f} %',
                     transform=ax.transAxes, ha='center', va='bottom', fontsize=10)

            buf = io.BytesIO()
            fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
            plt.close(fig)
            buf.seek(0)
            img = Image.open(buf)
            img.load()
            comp_frames.append(img.convert('RGB'))

        _save_gif(comp_frames, 'tracer_comparison.gif', width=650)
    else:
        def _render_field_gif(field_frames, gif_name, vmin, vmax, is_diff=False):
            frames = []
            for i in outlist:
                fig, ax = plt.subplots(figsize=[6, 8])
                ax.set_facecolor('k')

                im = ax.pcolormesh(
                    X, Z, field_frames[i],
                    cmap='coolwarm',
                    vmin=vmin, vmax=vmax,
                    shading='auto'
                )

                ax.set_xlabel('R')
                ax.set_ylabel('z')
                ax.set_aspect('equal')
                ax.set_xlim(0, 20)
                ax.set_ylim(0, 10)

                if 'STAR' in path:
                    ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
                    ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
                elif 'BH' in path:
                    ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
                    ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
                ax.legend(loc='upper right', labelcolor='w')

                divider = make_axes_locatable(ax)
                if is_diff:
                    cax = divider.append_axes("right", size="5%", pad=0.1)
                else:
                    cax = divider.append_axes("right", size="5%", pad=0.4)
                cbar = fig.colorbar(im, cax=cax)

                if is_diff:
                    cbar.set_label('A – B')
                else:
                    cbar.set_ticks([])
                    cax.text(0.5, 1.02, 'disk', transform=cax.transAxes,
                              ha='center', va='bottom', color='k', fontsize=10)
                    cax.text(0.5, -0.02, 'corona', transform=cax.transAxes,
                              ha='center', va='top', color='k', fontsize=10)

                buf = io.BytesIO()
                fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
                plt.close(fig)
                buf.seek(0)
                img = Image.open(buf)
                img.load()
                frames.append(img.convert('RGB'))

            _save_gif(frames, gif_name)

        if compare:
            gif_filename = 'tracer_diff.gif'
            _render_field_gif(tracer_frames, gif_filename, vmin=-1, vmax=1, is_diff=True)
        else:
            gif_filename = f'tracer_{mode}.gif'
            _render_field_gif(tracer_frames, gif_filename, vmin=0, vmax=1, is_diff=False)



def animate_panel(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist

    def profiles(D):
        rho = D.rho
        prs = D.prs
        vr = D.vx1
        vphi = D.vx3
        tr = D.tr1
        theta = D.x2
        r = D.x1

        sin_th = np.sin(theta)[None, :]
        l_specific = r[:, None] * np.sin(theta)[None, :] * vphi

        def weighted_avg(field, weight):
            num = np.trapezoid(field * weight * sin_th, theta, axis=1)
            den = np.trapezoid(weight * sin_th, theta, axis=1)
            return num / np.where(den == 0, np.nan, den)

        rho_disk = weighted_avg(rho, tr)
        rho_corona = weighted_avg(rho, 1 - tr)
        prs_disk = weighted_avg(prs, tr)
        prs_corona = weighted_avg(prs, 1 - tr)

        base = rho * vr * sin_th
        base_l = rho * vr * l_specific * sin_th
        Mdot_disk = 2 * np.pi * r**2 * np.trapezoid(base * tr, theta, axis=1)
        Mdot_corona = 2 * np.pi * r**2 * np.trapezoid(base * (1 - tr), theta, axis=1)
        Ldot_disk = 2 * np.pi * r**2 * np.trapezoid(base_l * tr, theta, axis=1)
        Ldot_corona = 2 * np.pi * r**2 * np.trapezoid(base_l * (1 - tr), theta, axis=1)

        return (r, rho_disk, rho_corona, prs_disk, prs_corona,
                Mdot_disk, Mdot_corona, Ldot_disk, Ldot_corona)

    r_ref = None
    rho_disk_frames, rho_corona_frames = {}, {}
    prs_disk_frames, prs_corona_frames = {}, {}
    Mdot_disk_frames, Mdot_corona_frames = {}, {}
    Ldot_disk_frames, Ldot_corona_frames = {}, {}

    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        r, rd, rc, pd, pc, md, mc, ld, lc = profiles(Di)
        r_ref = r
        rho_disk_frames[i] = rd
        rho_corona_frames[i] = rc
        prs_disk_frames[i] = pd
        prs_corona_frames[i] = pc
        Mdot_disk_frames[i] = md
        Mdot_corona_frames[i] = mc
        Ldot_disk_frames[i] = ld
        Ldot_corona_frames[i] = lc

    def shared_ylim(*frame_dicts, log=False):
        all_vals = np.concatenate([v[~np.isnan(v)] for d in frame_dicts for v in d.values()])
        if log:
            all_vals = all_vals[all_vals > 0]
            ymin, ymax = all_vals.min(), all_vals.max()
            pad = (ymax / ymin) ** 0.05
            return ymin / pad, ymax * pad
        ymin, ymax = all_vals.min(), all_vals.max()
        pad = 0.05 * (ymax - ymin)
        return ymin - pad, ymax + pad

    rho_ymin, rho_ymax = shared_ylim(rho_disk_frames, rho_corona_frames, log=True)
    prs_ymin, prs_ymax = shared_ylim(prs_disk_frames, prs_corona_frames, log=True)
    mdot_ymin, mdot_ymax = shared_ylim(Mdot_disk_frames, Mdot_corona_frames)
    ldot_ymin, ldot_ymax = shared_ylim(Ldot_disk_frames, Ldot_corona_frames)

    outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10

    frames = []
    for i in outlist:
        fig, axs = plt.subplots(4, 2, figsize=[7.5, 10], sharex=True)

        for col, frame_dict in [(0, rho_disk_frames), (1, rho_corona_frames)]:
            ax = axs[0, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, rho_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(rho_ymin, rho_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle\rho\rangle$')

        for col, frame_dict in [(0, prs_disk_frames), (1, prs_corona_frames)]:
            ax = axs[1, col]
            y = frame_dict[i]
            ax.fill_between(r_ref, y, prs_ymin, color='m', alpha=1/4, linewidth=0)
            ax.set_yscale('log')
            ax.set_ylim(prs_ymin, prs_ymax)
            if col == 0:
                ax.set_ylabel(r'$\langle P\rangle$')

        panel_spec = [
            (2, 0, Mdot_disk_frames,   r'$\dot{M}$',    mdot_ymin, mdot_ymax),
            (2, 1, Mdot_corona_frames, None,            mdot_ymin, mdot_ymax),
            (3, 0, Ldot_disk_frames,   r'$\dot{L}$',    ldot_ymin, ldot_ymax),
            (3, 1, Ldot_corona_frames, None,            ldot_ymin, ldot_ymax),
        ]
        for row, col, frame_dict, ylabel, ymin, ymax in panel_spec:
            ax = axs[row, col]
            y = frame_dict[i]
            ax.axhline(0, color='k', lw=0.8, ls='-')
            ax.fill_between(r_ref, y, 0, where=(y < 0), color='b', alpha=1/3, linewidth=0)
            ax.fill_between(r_ref, y, 0, where=(y >= 0), color='r', alpha=1/3, linewidth=0)
            ax.set_ylabel(ylabel)
            ax.set_ylim(ymin, ymax)

        for row in range(4):
            for col in range(2):
                ax = axs[row, col]
                ax.set_xscale('log')
                ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
                ax.xaxis.set_major_formatter(ScalarFormatter())
                ax.minorticks_off()
                ax.xaxis.set_minor_formatter(plt.NullFormatter())
                if row == 3:
                    ax.set_xlabel(r'$r$')

        axs[0, 0].set_title('Disk')
        axs[0, 1].set_title('Corona')
        for row in range(4):
            axs[row, 1].tick_params(labelleft=False)

        if 'STAR' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            axs[0, 1].plot([], [], ' ', label=f't = {times[i]:.0f}')
        axs[0, 1].legend(loc='upper right')

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    gif_path = f'{path}/Storage/panel.gif'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="750">'))

### 1. Plots

In [3]:
path = 'BH_VISC_HD/'
animate_density(path)
animate_velocity(path)
animate_panel(path)

In [4]:
path = 'BH_VISC_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [5]:
path = 'BH_VISC_RES_MHD/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [32]:
path = 'BH_VISC_HD/'
animate_tracer(path)

In [33]:
path = 'BH_VISC_MHD/'
animate_tracer(path)

In [34]:
path = 'BH_VISC_RES_MHD/'
animate_tracer(path)

In [40]:
path = 'BH_VISC_HD_TRC/'
animate_density(path)
animate_tracer(path, mode='truth')
animate_tracer(path, mode='recover')
animate_tracer(path, compare=True, panel=False)
animate_tracer(path, compare=True, panel=True)

In [41]:
path = 'BH_VISC_RES_MHD_TRC/'
animate_density(path)
animate_tracer(path, mode='truth')
animate_tracer(path, mode='recover')
animate_tracer(path, compare=True, panel=False)
animate_tracer(path, compare=True, panel=True)

In [5]:
path = 'BH_VISC_RES_MHD_TRC/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [19]:
path = 'BH_VISC_RES_MHD_TRC/'
animate_density(path)
animate_velocity(path)
animate_field(path)
animate_panel(path)

In [18]:
def animate_comparison(path1, path2):
    D1_last = pp.Load(nout='last', path=path1)
    unique_outs = list(D1_last.outlist)  # assumes matching nout indices in path2
    if 'STAR' in path1:
        times = D1_last.timelist / 62.8318
    elif 'BH' in path1:
        times = D1_last.timelist

    R, Theta = np.meshgrid(D1_last.x1, D1_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    rho_diff_frames = {}
    vx_diff_frames = {}
    vz_diff_frames = {}
    vr_diff_frames = {}

    for i in unique_outs:
        D1 = pp.Load(nout=i, path=path1)
        D2 = pp.Load(nout=i, path=path2)

        rho_diff = D1.rho - D2.rho
        rho_diff_frames[i] = np.where(mask, rho_diff, np.nan)

        vr1, vth1 = D1.vx1, D1.vx2
        vr2, vth2 = D2.vx1, D2.vx2
        vx1_ = vr1 * np.sin(Theta) + vth1 * np.cos(Theta)
        vz1_ = vr1 * np.cos(Theta) - vth1 * np.sin(Theta)
        vx2_ = vr2 * np.sin(Theta) + vth2 * np.cos(Theta)
        vz2_ = vr2 * np.cos(Theta) - vth2 * np.sin(Theta)

        vx_diff_frames[i] = np.where(mask, vx1_ - vx2_, np.nan)
        vz_diff_frames[i] = np.where(mask, vz1_ - vz2_, np.nan)
        vr_diff_frames[i] = np.where(mask, vr1 - vr2, np.nan)

    vmax_rho = np.nanmax([np.nanmax(np.abs(r)) for r in rho_diff_frames.values()])

    if 'STAR' in path1:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path1:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20

    def _save_gif(frames, gif_name, width=750):
        gif_path = f'{path1}/Storage/{gif_name}'
        frames[0].save(gif_path, format='GIF', save_all=True,
                        append_images=frames[1:], duration=150, loop=0)
        ipd.display(ipd.HTML(f'<img src="{gif_path}" width="{width}">'))

    # ---- density difference panel ----
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        im = ax.pcolormesh(X, Z, rho_diff_frames[i], cmap='coolwarm',
                            vmin=-vmax_rho, vmax=vmax_rho, shading='auto')
        ax.set_xlabel('R'); ax.set_ylabel('z'); ax.set_aspect('equal')
        ax.set_xlim(0, 20); ax.set_ylim(0, 10)
        if 'STAR' in path1:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path1:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho_1 - \rho_2$')
        buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig); buf.seek(0)
        frames.append(Image.open(buf).convert('RGB'))
    _save_gif(frames, 'density_comparison.gif')

    # ---- velocity difference quiver ----
    stride1, stride2 = 1, 1
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        im = ax.pcolormesh(X, Z, rho_diff_frames[i], cmap='gray',
                            vmin=-vmax_rho, vmax=vmax_rho, shading='auto')
        Xq = X[::stride1, ::stride2]; Zq = Z[::stride1, ::stride2]
        Uq = vx_diff_frames[i][::stride1, ::stride2]
        Wq = vz_diff_frames[i][::stride1, ::stride2]
        Vrq = vr_diff_frames[i][::stride1, ::stride2]
        colors = np.where(Vrq >= 0, 'r', 'b').ravel()
        ax.quiver(Xq, Zq, Uq, Wq, color=colors, scale_units='xy', angles='xy')
        ax.set_xlabel('R'); ax.set_ylabel('z'); ax.set_aspect('equal')
        ax.set_xlim(0, 20); ax.set_ylim(0, 10)
        if 'STAR' in path1:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path1:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        fig.colorbar(im, cax=cax, label=r'$\rho_1 - \rho_2$')
        buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig); buf.seek(0)
        frames.append(Image.open(buf).convert('RGB'))
    _save_gif(frames, 'velocity_comparison.gif')

In [13]:
path1 = 'BH_VISC_RES_MHD_TRC/'
path2 = 'BH_VISC_RES_MHD/'
animate_comparison(path1, path2)

In [19]:
D_test = pp.Load(nout='last', path=path1)
print(D_test.diskfrac)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 1. 1.]
 [0. 0. 0. ... 1. 1. 1.]
 [0. 0. 0. ... 1. 1. 1.]]


In [27]:
def animate_diskfraction(path, gif_name='disk_fraction.gif', width=750):
    """
    Renders and displays a single animated GIF of the computed disk fraction field.
    """
    median_factor = 5.0
    vphi_frac = 0.5
    blur_half_width = 1.1

    def rotation_mask(vphi, R, Theta):
        Rcyl = R * np.sin(Theta)
        vK = 1. / np.sqrt(Rcyl)
        return vphi > vphi_frac * vK

    def median_field(rho, vphi, R, Theta, factor):
        field = np.zeros_like(rho)
        for i in range(rho.shape[0]):
            med = np.median(rho[i, :])
            field[i, :] = (rho[i, :] > factor * med).astype(float)
        rot = rotation_mask(vphi, R, Theta)
        field[~rot] = 0.0
        return field

    def blur_field(rho, vphi, R, Theta, factor):
        field = median_field(rho, vphi, R, Theta, factor)
        size = int(2 * blur_half_width + 1)
        return uniform_filter(field, size=size, mode='nearest')

    # Load dataset metadata
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)

    if 'STAR' in path:
        times = D_last.timelist / 62.8318
        pad_count = 10
    elif 'BH' in path:
        times = D_last.timelist
        pad_count = 20
    else:
        times = D_last.timelist
        pad_count = 10

    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)

    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)

    # Compute disk fraction frames
    disk_frac_frames = {}
    for i in unique_outs:
        Di = pp.Load(nout=i, path=path)
        blur = blur_field(Di.rho, Di.vx3, R, Theta, median_factor)
        disk_frac_frames[i] = np.where(mask, blur, np.nan)

    # Apply boundary frame padding for smooth looping
    outlist = [unique_outs[0]] * pad_count + unique_outs + [unique_outs[-1]] * pad_count

    # Render animation frames
    frames = []
    for i in outlist:
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')

        im = ax.pcolormesh(
            X, Z, disk_frac_frames[i],
            cmap='coolwarm',
            vmin=0, vmax=1,
            shading='auto'
        )

        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)

        # Plot central object boundary
        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')

        # Colorbar configuration
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.4)
        cbar = fig.colorbar(im, cax=cax)

        cbar.set_ticks([])
        cax.text(0.5, 1.02, 'disk', transform=cax.transAxes,
                  ha='center', va='bottom', color='k', fontsize=10)
        cax.text(0.5, -0.02, 'corona', transform=cax.transAxes,
                  ha='center', va='top', color='k', fontsize=10)

        # Buffer frame into RAM
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    # Save and render GIF
    gif_path = f'{path}/Storage/{gif_name}'
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    ipd.display(ipd.HTML(f'<img src="{gif_path}" width="{width}">'))

In [28]:
path = 'BH_VISC_HD_TRC/'
D_test = pp.Load(nout='last', path=path)
print(D_test)
animate_tracer(path)
animate_diskfraction(path)


        Load class.
        It loads the data.

        File properties:
        - Current path loaded (pathdir)      BH_VISC_HD_TRC
        - Format loaded       (format)       dbl

        Simulation properties
        - Dimensions    (dim)      2
        - Geometry      (geom)     SPHERICAL
        - Grid size     (gridsize) 4050
        - Grid shape    (nshp)     (45, 90)
        - Output loaded (nout)     200
        - Time loaded   (ntime)    1000.0

        Public attributes available:
        - Number of cells in each direction ['nx1', 'nx2', 'nx3']
        - Grid values (cell center)         ['x1', 'x2', 'x3']
        - Grid values (face center)         ['x1r', 'x2r', 'x3r']
        - Cells size                        ['dx1', 'dx2', 'dx3']
        - Time attributes                   ['outlist', 'timelist']
        - Projections ['x1c', 'x2c', 'x1rc', 'x2rc']

        Variables available:
        ['rho', 'vx1', 'vx2', 'vx3', 'prs', 'tr1', 'nu', 'num', 'Te', 'diskfrac']
       

In [17]:
# ============================================================================
# CONFIG
# ============================================================================
MEDIAN_FACTOR = 5.0
VPHI_FRAC = 0.5
BLUR_WIDTH_MAX = 1.0  # steady-state angular blur half-width (from CAMK validation)
TAU_BLUR = 10.0      # timescale for spatial blur to ramp up (steps; placeholder)
TAU_EMA = 10.0       # timescale for temporal EMA to ramp down (steps; placeholder)
ALPHA_MAX = 0.9      # initial (fast-tracking) EMA rate
ALPHA_MIN = 0.1      # steady-state EMA rate
RHOC_RESCALE_EVERY_N = 10**12  # recompute RHOC_eff every N steps

# ============================================================================
# HELPER: Rotation mask (unchanged)
# ============================================================================
def rotation_mask(vphi, R, Theta):
    """Returns True where vphi > vphi_frac * v_Kepler."""
    Rcyl = R * np.sin(Theta)
    vK = 1.0 / np.sqrt(np.maximum(Rcyl, 1e-12))
    return vphi > VPHI_FRAC * vK

# ============================================================================
# HELPER: Hard density-based classification at a single radius
# ============================================================================
def density_threshold_at_radius(rho_slice, median_factor):
    """
    Given a 1D slice of density at fixed radius across all theta,
    return a binary field: 1 where rho > median_factor * median(rho).
    """
    med = np.median(rho_slice)
    return (rho_slice > median_factor * med).astype(float)

# ============================================================================
# HELPER: Rescaled corona reference profile
# ============================================================================
def corona_reference_profile(R, RHOC_eff):
    """
    Analytic corona density: rho_ref = RHOC_eff * R^(-1.5).
    """
    return RHOC_eff * np.power(R, -1.5)

# ============================================================================
# MAIN CLASSIFIER: Density + Rotation AND-combined, with ramping blur
# ============================================================================
class DiskCoronaClassifier:
    def __init__(self, RHOC_init, times_list):
        """
        RHOC_init: initial RHOC parameter value
        times_list: array of simulation times at each output (for ramp functions)
        """
        self.RHOC_eff = RHOC_init
        self.times = times_list
        self.last_corona_mask = None
        self.step_index = 0
        
    def get_blur_width(self, time):
        """Spatial blur half-width, ramping from 0 to BLUR_WIDTH_MAX."""
        return BLUR_WIDTH_MAX * (1.0 - np.exp(-time / TAU_BLUR))
    
    def get_ema_alpha(self, time):
        """Temporal EMA rate, ramping from ALPHA_MAX down to ALPHA_MIN."""
        return ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * np.exp(-time / TAU_EMA)
    
    def rescale_RHOC_eff(self, rho, R, Theta):
        """
        Using cells currently labeled corona (from last classification),
        compute mean of rho / R^(-1.5) over those cells -> new RHOC_eff.
        Skip if corona-cell count < 10% of total (too noisy).
        """
        if self.last_corona_mask is None:
            return  # First call, no prior classification yet
        
        corona_cells = self.last_corona_mask > 0.5
        if np.sum(corona_cells) < 0.1 * corona_cells.size:
            return  # Too few corona cells, skip rescale
        
        rho_ref_inverse = np.power(R, 1.5)  # 1 / R^(-1.5)
        ratio = rho[corona_cells] * rho_ref_inverse[corona_cells]
        self.RHOC_eff = np.mean(ratio)
    
    def classify(self, rho, vphi, R, Theta, time, rescale_this_step=False):
        """
        Full disk/corona classification pipeline:
        1. Rescale RHOC_eff if requested (every N steps)
        2. Compute raw density+rotation mask
        3. Spatial blur (ramping width)
        4. Temporal EMA relaxation (ramping alpha)
        Returns: disk_frac field (0=corona, 1=disk)
        """
        # Step 1: Rescale RHOC_eff from prior corona region
        if rescale_this_step:
            self.rescale_RHOC_eff(rho, R, Theta)
        
        # Step 2: Hard classification (density vs rescaled corona ref + rotation veto)
        rho_ref = corona_reference_profile(R, self.RHOC_eff)
        raw_mask = (rho > MEDIAN_FACTOR * rho_ref).astype(float)
        rot = rotation_mask(vphi, R, Theta)
        raw_mask[~rot] = 0.0  # AND with rotation
        
        # Step 3: Spatial blur (ramping width)
        blur_width = self.get_blur_width(time)
        size = 2 * blur_width + 1
        smooth_mask = uniform_filter(raw_mask, size=size, mode='nearest')
        
        # Step 4: Temporal EMA (ramping alpha)
        if self.last_corona_mask is None:
            # First call: seed directly from smooth_mask (no blending against zero)
            disk_frac = smooth_mask
        else:
            alpha = self.get_ema_alpha(time)
            disk_frac = alpha * smooth_mask + (1.0 - alpha) * self.last_corona_mask
        
        self.last_corona_mask = disk_frac
        self.step_index += 1
        return disk_frac

# ============================================================================
# GIF RENDERING HELPER
# ============================================================================
def _save_gif(frames, gif_name, gif_path, width=750):
    """Save list of PIL Images as an animated GIF."""
    frames[0].save(
        gif_path,
        format='GIF',
        save_all=True,
        append_images=frames[1:],
        duration=150,
        loop=0
    )
    print(f"Saved: {gif_path}")

# ============================================================================
# INDIVIDUAL GIF RENDER FUNCTIONS
# ============================================================================

def animate_tracer(path, D_last, outlist, times, R, Theta, X, Z, mask):
    """Render and save tracer_truth.gif from real tracer field (tr1)."""
    frames = []
    for i in outlist:
        Di = pp.Load(nout=i, path=path)
        truth = Di.tr1
        
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        
        im = ax.pcolormesh(
            X, Z, np.where(mask, truth, np.nan),
            cmap='coolwarm', vmin=0, vmax=1,
            shading='auto'
        )
        
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)
        
        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.4)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_ticks([])
        cax.text(0.5, 1.02, 'disk', transform=cax.transAxes,
                  ha='center', va='bottom', color='k', fontsize=10)
        cax.text(0.5, -0.02, 'corona', transform=cax.transAxes,
                  ha='center', va='top', color='k', fontsize=10)
        
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/tracer_truth.gif'
    _save_gif(frames, 'tracer_truth', gif_path)

def animate_classifier(path, D_last, outlist, times, R, Theta, X, Z, mask, classifier):
    """Render and save tracer_blur.gif from new rescaled-corona classifier."""
    frames = []
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        classified = classifier.classify(Di.rho, Di.vx3, R, Theta, times[i], rescale_this_step)
        
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        
        im = ax.pcolormesh(
            X, Z, np.where(mask, classified, np.nan),
            cmap='coolwarm', vmin=0, vmax=1,
            shading='auto'
        )
        
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)
        
        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.4)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_ticks([])
        cax.text(0.5, 1.02, 'disk', transform=cax.transAxes,
                  ha='center', va='bottom', color='k', fontsize=10)
        cax.text(0.5, -0.02, 'corona', transform=cax.transAxes,
                  ha='center', va='top', color='k', fontsize=10)
        
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/tracer_blur.gif'
    _save_gif(frames, 'tracer_blur', gif_path)

def animate_difference(path, D_last, outlist, times, R, Theta, X, Z, mask, classifier):
    """Render and save tracer_blur_truth.gif: difference between classifier and real tracer."""
    frames = []
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        classified = classifier.classify(Di.rho, Di.vx3, R, Theta, times[i], rescale_this_step)
        truth = Di.tr1
        diff = classified - truth
        
        fig, ax = plt.subplots(figsize=[6, 8])
        ax.set_facecolor('k')
        
        im = ax.pcolormesh(
            X, Z, np.where(mask, diff, np.nan),
            cmap='coolwarm', vmin=-1, vmax=1,
            shading='auto'
        )
        
        ax.set_xlabel('R')
        ax.set_ylabel('z')
        ax.set_aspect('equal')
        ax.set_xlim(0, 20)
        ax.set_ylim(0, 10)
        
        if 'STAR' in path:
            ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right', labelcolor='w')
        
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label('Classifier – Truth')
        
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/tracer_blur_truth.gif'
    _save_gif(frames, 'tracer_blur_truth', gif_path)

def animate_comparison(path, D_last, outlist, times, R, Theta, classifier):
    """Render and save tracer_comparison.gif: per-radius disagreement metric."""
    frames = []
    x1 = D_last.x1
    n1 = D_last.rho.shape[0]
    
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        classified = classifier.classify(Di.rho, Di.vx3, R, Theta, times[i], rescale_this_step)
        truth = Di.tr1
        diff = classified - truth
        abs_diff = np.abs(diff)
        n_theta = abs_diff.shape[1]
        
        disagreement_per_radius = np.sum(abs_diff, axis=1) / n_theta
        disagreement_pct = disagreement_per_radius * 100.0
        total_disagreement_pct = (np.sum(disagreement_per_radius) / n1) * 100.0
        total_agreement_pct = 100.0 - total_disagreement_pct
        
        fig, ax = plt.subplots(figsize=[7, 4])
        ax.fill_between(x1, disagreement_pct, -5, color='m', alpha=1/4, linewidth=0)
        ax.set_xscale('log')
        ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.minorticks_off()
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.set_yticks([0, 5, 10, 15, 20])
        ax.set_xlabel('R')
        ax.set_ylabel('|Classifier – Truth| %')
        ax.set_ylim(-5, 20)
        
        if 'STAR' in path:
            ax.plot([], [], ' ', label=f't = {times[i]:.1f}')
        elif 'BH' in path:
            ax.plot([], [], ' ', label=f't = {times[i]:.0f}')
        ax.legend(loc='upper right')
        
        ax.text(0.5, 0.05, f'Agreement = {total_agreement_pct:.1f} %',
                 transform=ax.transAxes, ha='center', va='bottom', fontsize=10)
        
        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))
    
    gif_path = f'{path}/Storage/tracer_comparison.gif'
    _save_gif(frames, 'tracer_comparison', gif_path, width=650)

# ============================================================================
# MAIN ENTRY POINT
# ============================================================================
def animate_all(path):
    """
    Load simulation, initialize classifier, generate all four GIFs.
    """
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    
    if 'STAR' in path:
        times = D_last.timelist / 62.8318
    elif 'BH' in path:
        times = D_last.timelist
    else:
        times = D_last.timelist
    
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X = R * np.sin(Theta)
    Z = R * np.cos(Theta)
    
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    
    # Construct output list with padding
    if 'STAR' in path:
        outlist = [unique_outs[0]] * 10 + unique_outs + [unique_outs[-1]] * 10
    elif 'BH' in path:
        outlist = [unique_outs[0]] * 20 + unique_outs + [unique_outs[-1]] * 20
    else:
        outlist = unique_outs
    
    # Initialize classifier with RHOC from last snapshot
    RHOC_init = 0.01  # or g_inputParam[RHOC] equivalent
    classifier = DiskCoronaClassifier(RHOC_init, times)
    
    print("Rendering tracer_truth.gif...")
    animate_tracer(path, D_last, unique_outs, times, R, Theta, X, Z, mask)
    
    print("Rendering tracer_blur.gif (new classifier)...")
    animate_classifier(path, D_last, unique_outs, times, R, Theta, X, Z, mask, classifier)
    
    print("Rendering tracer_blur_truth.gif (difference)...")
    animate_difference(path, D_last, unique_outs, times, R, Theta, X, Z, mask, classifier)
    
    print("Rendering tracer_comparison.gif (per-radius agreement)...")
    animate_comparison(path, D_last, unique_outs, times, R, Theta, classifier)
    
    print("Done.")

# ============================================================================
# USAGE: animate_all('/path/to/sim')
# ============================================================================

In [18]:
animate_all('BH_VISC_HD')

Rendering tracer_truth.gif...
Saved: BH_VISC_HD/Storage/tracer_truth.gif
Rendering tracer_blur.gif (new classifier)...
Saved: BH_VISC_HD/Storage/tracer_blur.gif
Rendering tracer_blur_truth.gif (difference)...
Saved: BH_VISC_HD/Storage/tracer_blur_truth.gif
Rendering tracer_comparison.gif (per-radius agreement)...
Saved: BH_VISC_HD/Storage/tracer_comparison.gif
Done.


In [ ]:
# ============================================================================
# CONFIG
# ============================================================================
MEDIAN_FACTOR = 5.0
VPHI_FRAC = 0.5
BLUR_WIDTH_MAX = 1.1
TAU_BLUR = 200.0
TAU_EMA = 200.0
ALPHA_MAX = 0.95
ALPHA_MIN = 0.08
RHOC_RESCALE_EVERY_N = 1

_RHOC_eff = None
_last_disk_frac = None


def rotation_mask(vphi, R, Theta):
    Rcyl = R * np.sin(Theta)
    vK = 1.0 / np.sqrt(np.maximum(Rcyl, 1e-12))
    return vphi > VPHI_FRAC * vK


def corona_reference_profile(R, RHOC_eff):
    """rho_ref = RHOC_eff * R^(-1.5)."""
    return RHOC_eff * np.power(R, -1.5)


def rescale_RHOC_eff(rho, R, last_disk_frac, RHOC_current):
    """
    Rescale in log-space so the R^(-1.5) divergence near the inner
    boundary doesn't multiplicatively dominate the mean ratio:

        log(RHOC_eff_new) = mean[ log(rho) + 1.5*log(R) ]  over corona cells
    """
    if last_disk_frac is None:
        return RHOC_current

    corona_cells = last_disk_frac < 0.5
    if np.sum(corona_cells) < 0.1 * corona_cells.size:
        return RHOC_current

    log_rho = np.log(np.maximum(rho[corona_cells], 1e-30))
    log_R = np.log(R[corona_cells])
    log_RHOC_eff = np.mean(log_rho + 1.5 * log_R)
    return np.exp(log_RHOC_eff)


def get_blur_width(time):
    return BLUR_WIDTH_MAX * (1.0 - np.exp(-time / TAU_BLUR))


def get_ema_alpha(time):
    return ALPHA_MIN + (ALPHA_MAX - ALPHA_MIN) * np.exp(-time / TAU_EMA)


def classify_disk_corona(rho, vphi, R, Theta, time, RHOC_current, last_disk_frac,
                          rescale_this_step):
    """Returns (disk_frac, RHOC_eff_used)."""
    RHOC_eff = RHOC_current
    if rescale_this_step:
        RHOC_eff = rescale_RHOC_eff(rho, R, last_disk_frac, RHOC_current)

    rho_ref = corona_reference_profile(R, RHOC_eff)
    raw_mask = (rho > MEDIAN_FACTOR * rho_ref).astype(float)
    rot = rotation_mask(vphi, R, Theta)
    raw_mask[~rot] = 0.0

    blur_width = get_blur_width(time)
    size = 2 * blur_width + 1
    smooth_mask = uniform_filter(raw_mask, size=size, mode='nearest')

    if last_disk_frac is None:
        disk_frac = smooth_mask
    else:
        alpha = get_ema_alpha(time)
        disk_frac = alpha * smooth_mask + (1.0 - alpha) * last_disk_frac

    return disk_frac, RHOC_eff


def reset_classifier_state(RHOC_init):
    """Call once before iterating over outputs."""
    global _RHOC_eff, _last_disk_frac
    _RHOC_eff = RHOC_init
    _last_disk_frac = None


def _save_gif(frames, gif_path, width=750):
    frames[0].save(
        gif_path, format='GIF', save_all=True,
        append_images=frames[1:], duration=150, loop=0
    )
    print(f"Saved: {gif_path}")


def _draw_field_frame(X, Z, field, path, time_label, vmin, vmax, is_diff):
    fig, ax = plt.subplots(figsize=[6, 8])
    ax.set_facecolor('k')
    im = ax.pcolormesh(X, Z, field, cmap='coolwarm', vmin=vmin, vmax=vmax, shading='auto')

    ax.set_xlabel('R')
    ax.set_ylabel('z')
    ax.set_aspect('equal')
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 10)

    if 'STAR' in path:
        ax.add_patch(Wedge((0.0, 0.0), 1.0, 0.0, 90, facecolor='w', edgecolor=None))
    elif 'BH' in path:
        ax.add_patch(Wedge((0.0, 0.0), 2.1, 0.0, 90, facecolor='k', edgecolor=None))
    ax.plot([], [], ' ', label=time_label)
    ax.legend(loc='upper right', labelcolor='w')

    divider = make_axes_locatable(ax)
    if is_diff:
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_label('A – B')
    else:
        cax = divider.append_axes("right", size="5%", pad=0.4)
        cbar = fig.colorbar(im, cax=cax)
        cbar.set_ticks([])
        cax.text(0.5, 1.02, 'disk', transform=cax.transAxes, ha='center', va='bottom', color='k', fontsize=10)
        cax.text(0.5, -0.02, 'corona', transform=cax.transAxes, ha='center', va='top', color='k', fontsize=10)

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf)
    img.load()
    return img.convert('RGB')


def _time_label(path, times, i):
    if 'STAR' in path:
        return f't = {times[i]:.1f}'
    return f't = {times[i]:.0f}'


def _load_common(path):
    D_last = pp.Load(nout='last', path=path)
    unique_outs = list(D_last.outlist)
    times = D_last.timelist / 62.8318 if 'STAR' in path else D_last.timelist
    R, Theta = np.meshgrid(D_last.x1, D_last.x2, indexing='ij')
    X, Z = R * np.sin(Theta), R * np.cos(Theta)
    mask = (X >= 0) & (X <= 21) & (Z >= 0) & (Z <= 11)
    pad = 10 if 'STAR' in path else 20
    outlist = [unique_outs[0]] * pad + unique_outs + [unique_outs[-1]] * pad
    return D_last, unique_outs, times, R, Theta, X, Z, mask, outlist


# ============================================================================
# FOUR SEPARATE GIF FUNCTIONS
# ============================================================================

def animate_tracer(path):
    """tracer_truth.gif -- real tracer field only."""
    D_last, unique_outs, times, R, Theta, X, Z, mask, outlist = _load_common(path)

    frames = []
    for i in outlist:
        Di = pp.Load(nout=i, path=path)
        truth = np.where(mask, Di.tr1, np.nan)
        frames.append(_draw_field_frame(X, Z, truth, path, _time_label(path, times, i), 0, 1, False))

    _save_gif(frames, f'{path}/Storage/tracer_truth.gif')


def animate_classifier(path, RHOC_init):
    """tracer_blur.gif -- new rescaled-corona classifier."""
    D_last, unique_outs, times, R, Theta, X, Z, mask, outlist = _load_common(path)
    reset_classifier_state(RHOC_init)
    global _RHOC_eff, _last_disk_frac

    frames = []
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        disk_frac, _RHOC_eff = classify_disk_corona(
            Di.rho, Di.vx3, R, Theta, times[i], _RHOC_eff, _last_disk_frac, rescale_this_step
        )
        _last_disk_frac = disk_frac
        field = np.where(mask, disk_frac, np.nan)
        frames.append(_draw_field_frame(X, Z, field, path, _time_label(path, times, i), 0, 1, False))

    _save_gif(frames, f'{path}/Storage/tracer_blur.gif')


def animate_difference(path, RHOC_init):
    """tracer_blur_truth.gif -- classifier minus real tracer."""
    D_last, unique_outs, times, R, Theta, X, Z, mask, outlist = _load_common(path)
    reset_classifier_state(RHOC_init)
    global _RHOC_eff, _last_disk_frac

    frames = []
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        disk_frac, _RHOC_eff = classify_disk_corona(
            Di.rho, Di.vx3, R, Theta, times[i], _RHOC_eff, _last_disk_frac, rescale_this_step
        )
        _last_disk_frac = disk_frac
        diff = np.where(mask, disk_frac - Di.tr1, np.nan)
        frames.append(_draw_field_frame(X, Z, diff, path, _time_label(path, times, i), -1, 1, True))

    _save_gif(frames, f'{path}/Storage/tracer_blur_truth.gif')


def animate_comparison(path, RHOC_init):
    """tracer_comparison.gif -- per-radius disagreement %."""
    D_last, unique_outs, times, R, Theta, X, Z, mask, outlist = _load_common(path)
    x1 = D_last.x1
    n1 = D_last.rho.shape[0]
    reset_classifier_state(RHOC_init)
    global _RHOC_eff, _last_disk_frac

    frames = []
    for idx, i in enumerate(outlist):
        Di = pp.Load(nout=i, path=path)
        rescale_this_step = (idx % RHOC_RESCALE_EVERY_N == 0)
        disk_frac, _RHOC_eff = classify_disk_corona(
            Di.rho, Di.vx3, R, Theta, times[i], _RHOC_eff, _last_disk_frac, rescale_this_step
        )
        _last_disk_frac = disk_frac

        abs_diff = np.abs(disk_frac - Di.tr1)
        n_theta = abs_diff.shape[1]
        disagreement_pct = (np.sum(abs_diff, axis=1) / n_theta) * 100.0
        total_agreement_pct = 100.0 - (np.sum(disagreement_pct) / n1)

        fig, ax = plt.subplots(figsize=[7, 4])
        ax.fill_between(x1, disagreement_pct, -5, color='m', alpha=1/4, linewidth=0)
        ax.set_xscale('log')
        ax.set_xticks([2, 4, 6, 8, 10, 20, 30])
        ax.xaxis.set_major_formatter(ScalarFormatter())
        ax.minorticks_off()
        ax.xaxis.set_minor_formatter(plt.NullFormatter())
        ax.set_yticks([0, 5, 10, 15, 20])
        ax.set_xlabel('R')
        ax.set_ylabel('|A – B| %')
        ax.set_ylim(-5, 20)
        ax.plot([], [], ' ', label=_time_label(path, times, i))
        ax.legend(loc='upper right')
        ax.text(0.5, 0.05, f'1 – |A – B| = {total_agreement_pct:.1f} %',
                transform=ax.transAxes, ha='center', va='bottom', fontsize=10)

        buf = io.BytesIO()
        fig.savefig(buf, format='png', dpi=200, bbox_inches='tight')
        plt.close(fig)
        buf.seek(0)
        img = Image.open(buf)
        img.load()
        frames.append(img.convert('RGB'))

    _save_gif(frames, f'{path}/Storage/tracer_comparison.gif', width=650)

In [29]:
animate_tracer('BH_VISC_HD_TRC')
animate_classifier('BH_VISC_HD_TRC', 0.01)
animate_difference('BH_VISC_HD_TRC', 0.01)
animate_comparison('BH_VISC_HD_TRC')

Saved: BH_VISC_HD_TRC/Storage/tracer_truth.gif
Saved: BH_VISC_HD_TRC/Storage/tracer_blur.gif
Saved: BH_VISC_HD_TRC/Storage/tracer_blur_truth.gif


TypeError: animate_comparison() missing 1 required positional argument: 'RHOC_init'